# TinyStories Scale-Up v3: Structured V_θ (SQ3) for PARF vs FockPARF vs Hybrids

## Motivation

On TinyShakespeare (~1M tokens), the SQ3 structured V_θ
(`MixtureQuadraticVTheta`, K=4 diagonal quadratic wells) matches the MLP
V_θ baseline at 184.5 vs 185.5 PPL with **zero autograd.grad cost** and
exposes the K*=4 attractor basin structure analytically.

This v3 notebook replaces the MLP V_θ in all four cells (S1–S4) with
`MixtureQuadraticVTheta(K=4)` and scales up to TinyStories (~5M tokens).
The outputs go to `semsimula_tinystories_v3/` on GDrive to avoid overwriting
v2 MLP-V_θ results.

## v3 changes (vs v2 MLP V_θ run in `semsimula_tinystories_v2/`)

| Parameter | v2 (all cells) | v3 (all cells) | Reason |
|---|---|---|---|
| `v_theta_kind` | `mlp` (v_hidden=1024, 3-layer) | **`sq3`** (K=4 mixture) | SQ3 matches MLP PPL at ~20% params; analytical gradient |
| `lambda_V` | 1e-2 | **1e-2** | unchanged — PR2 optimal |
| Steps | 16 000 | **16 000** | unchanged |
| GDrive dir | `semsimula_tinystories_v2` | **`semsimula_tinystories_v3`** | preserve v2 results |

## Recommended run order

**Start with S3** (Hybrid FockPARF+Attn) — highest-value experiment.
Then S2, S1, S4.

## Cell structure

| Cell | Architecture | V_θ | lambda_V | Priority |
|------|-------------|-----|----------|----------|
| `S3` | **Hybrid FockPARF+Attn** (k=4 attn + L=4 FockPARF, M=32) | SQ3 K=4 | 1e-2 | **1 — run first** |
| `S2` | Regularised FockPARF (d=256, L=8, M=32) | SQ3 K=4 | 1e-2 | 2 |
| `S1` | Regularised PARF (d=256, L=8) | SQ3 K=4 | 1e-2 | 3 |
| `S4` | Hybrid SPLM+Attn reference (k=4 attn + L=4 SPLM) | SQ3 K=4 | 1e-2 | 4 |

Reference baselines:
- Matched attention (8L GPT-2): **7.81 PPL**
- PARF P10g unregularised (16k steps): **26.42 PPL**
- **v2 Reg PARF   (MLP V_θ, 16k steps): TBD**  ← v2 results for comparison
- **v2 Reg FockPARF (MLP V_θ, 16k steps): TBD** ← v2 results for comparison
- **TinyShakespeare SQ3 PARF (4k steps, d=128): 184.5 PPL**

Outputs go to `semsimula_tinystories_v3/` on GDrive.
Run one `CELL` at a time; outputs persist across sessions.


## 0. Environment setup + cell selector

In [ ]:
CELL = 'S3'       # one of: 'S1' | 'S2' | 'S3' | 'S4'  -- run S3 first!
SEED = 0

REPO_URL        = 'https://github.com/dimitarpg13/semsimula-paper.git'
REPO_BRANCH     = 'main'
COLAB_REPO_PATH = '/content/semsimula-paper'
GDRIVE_OUT_REL  = 'semsimula_tinystories_v3'

import os, sys, shutil, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print(f'IN_COLAB = {IN_COLAB}')


def _sh(cmd: str) -> None:
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f'command failed (exit {r.returncode}): {cmd}')


if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    GDRIVE_OUT = Path('/content/drive/MyDrive') / GDRIVE_OUT_REL
    GDRIVE_OUT.mkdir(parents=True, exist_ok=True)
    print(f'GDrive output root = {GDRIVE_OUT}')

    REPO_ROOT = Path(COLAB_REPO_PATH)
    if not (REPO_ROOT / '.git').exists():
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        _sh(
            f'git clone --depth 1 --branch {REPO_BRANCH} '
            f'{REPO_URL} {REPO_ROOT}'
        )
    else:
        try:
            _sh(f'git -C {REPO_ROOT} fetch --depth 1 origin {REPO_BRANCH}')
            _sh(f'git -C {REPO_ROOT} reset --hard origin/{REPO_BRANCH}')
        except RuntimeError as e:
            print(f'WARNING: could not refresh repo ({e}); using existing checkout.')

    DATA_CACHE = GDRIVE_OUT / 'data'
    DATA_CACHE.mkdir(exist_ok=True)
    repo_data_dir = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    if repo_data_dir.is_symlink():
        repo_data_dir.unlink()
    elif repo_data_dir.is_dir():
        shutil.rmtree(repo_data_dir)
    repo_data_dir.symlink_to(DATA_CACHE)
    print(f'data/ -> {DATA_CACHE}')

    _sh('pip install -q transformers huggingface_hub pyarrow')
else:
    REPO_ROOT = Path('.').resolve()
    while not (REPO_ROOT / '.git').exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent
    GDRIVE_OUT = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'parf' / 'results' / 'tinystories_parf_fockparf'
    GDRIVE_OUT.mkdir(parents=True, exist_ok=True)

SARF_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch'
sys.path.insert(0, str(SARF_DIR))
sys.path.insert(0, str(SARF_DIR / 'parf'))
sys.path.insert(0, str(SARF_DIR / 'sarf_mass_variant'))
sys.path.insert(0, str(SARF_DIR / 'energetic_minima'))

RESULTS_ROOT = GDRIVE_OUT
RUN_DIR = RESULTS_ROOT / CELL / f'seed{SEED}'
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f'Run output dir = {RUN_DIR}')

## 1. GPU check

In [ ]:
import torch
import numpy as np

if torch.cuda.is_available():
    device = 'cuda'
    props = torch.cuda.get_device_properties(0)
    total_memory = props.total_memory / 1e9
    print(f'GPU: {props.name}  ({total_memory:.1f} GB)')
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    print('TF32 disabled (SPLM autograd.grad sensitivity)')
elif torch.backends.mps.is_available():
    device = 'mps'
    print('Using MPS (Apple Silicon)')
else:
    device = 'cpu'
    print('WARNING: no GPU detected; training will be very slow')

print(f'device = {device}')
rng = np.random.default_rng(SEED)

## 2. Experiment recipes

In [ ]:
SCALE_RECIPES = {
    'S1': {
        'desc': 'Regularised PARF + SQ3 V_theta on TinyStories',
        'model': 'parf',
        'd': 256, 'L': 8,
        'v_theta_kind': 'sq3', 'v_theta_K': 4,   # structured V_theta
        'v_hidden': 1024, 'v_depth': 3,           # ignored when v_theta_kind='sq3'
        'n_attn': 0, 'n_head': 0, 'mlp_mult': 0,
        'n_registers': 0, 'creation_gate_hidden': 0,
        'top_k': 4, 'score_head_hidden': 32,
        'lambda_v': 1e-2, 'steps': 16000,
        'batch': 16, 'block': 512,
        'init_gamma': 0.10, 'fixed_gamma': None,
    },
    'S2': {
        'desc': 'Regularised FockPARF + SQ3 V_theta on TinyStories',
        'model': 'fock_parf',
        'd': 256, 'L': 8,
        'v_theta_kind': 'sq3', 'v_theta_K': 4,
        'v_hidden': 1024, 'v_depth': 3,
        'n_attn': 0, 'n_head': 0, 'mlp_mult': 0,
        'n_registers': 32, 'creation_gate_hidden': 64,
        'top_k': 4, 'score_head_hidden': 32,
        'lambda_v': 1e-2, 'steps': 16000,
        'batch': 16, 'block': 512,
        'init_gamma': 0.10, 'fixed_gamma': None,
    },
    'S3': {
        'desc': 'Hybrid FockPARF+Attn + SQ3 V_theta on TinyStories',
        'model': 'hybrid_fock_parf',
        'd': 256, 'L': 4,
        'v_theta_kind': 'sq3', 'v_theta_K': 4,
        'v_hidden': 1024, 'v_depth': 3,
        'n_attn': 4, 'n_head': 4, 'mlp_mult': 4,
        'n_registers': 32, 'creation_gate_hidden': 64,
        'top_k': 4, 'score_head_hidden': 32,
        'lambda_v': 1e-2, 'steps': 16000,
        'batch': 8, 'block': 512,
        'init_gamma': 0.10, 'fixed_gamma': None,
    },
    'S4': {
        'desc': 'Hybrid SPLM+Attn + SQ3 V_theta on TinyStories',
        'model': 'hybrid_splm',
        'd': 256, 'L': 4,
        'v_theta_kind': 'sq3', 'v_theta_K': 4,
        'v_hidden': 1024, 'v_depth': 3,
        'n_attn': 4, 'n_head': 4, 'mlp_mult': 4,
        'n_registers': 0, 'creation_gate_hidden': 0,
        'top_k': 0, 'score_head_hidden': 0,
        'lambda_v': 1e-2, 'steps': 16000,
        'batch': 8, 'block': 512,
        'init_gamma': 0.10, 'fixed_gamma': 0.10,
    },
}
if CELL not in SCALE_RECIPES:
    raise ValueError(f'CELL must be one of {list(SCALE_RECIPES)}; got {CELL!r}')

recipe = SCALE_RECIPES[CELL]
LAMBDA_V       = recipe['lambda_v']
STEPS          = recipe['steps']
D              = recipe['d']
L              = recipe['L']
V_THETA_KIND   = recipe.get('v_theta_kind', 'mlp')
V_THETA_K      = recipe.get('v_theta_K', 4)
V_HIDDEN       = recipe['v_hidden']
V_DEPTH        = recipe['v_depth']
N_ATTN         = recipe['n_attn']
N_HEAD         = recipe['n_head']
MLP_MULT       = recipe['mlp_mult']
N_REGISTERS    = recipe['n_registers']
CREATION_GATE_HIDDEN = recipe['creation_gate_hidden']
TOP_K          = recipe['top_k']
SCORE_HEAD_HIDDEN = recipe['score_head_hidden']
BATCH          = recipe['batch']
BLOCK          = recipe['block']
INIT_GAMMA     = recipe.get('init_gamma', 0.15)
FIXED_GAMMA    = recipe.get('fixed_gamma', None)
MODEL_KIND     = recipe['model']

VOCAB_SIZE  = 50257
MAX_LEN     = 1024
DT          = 1.0
V_PHI_KIND  = 'structural'
V_PHI_D_TYPE = 32
V_PHI_D_ANGLE = 16
V_PHI_PHI_HIDDEN = 16
V_PHI_THETA_HIDDEN = 16
V_PHI_MLP_HIDDEN = 32
GUMBEL_TAU_INIT = 1.0
GUMBEL_TAU_MIN = 0.1
LR = 5e-4
WD = 0.01
WARMUP = 800
GRAD_CLIP = 1.0
EVAL_INTERVAL = 400
EVAL_ITERS = 40
LOG_INTERVAL = 50
STACK_DISCIPLINE = True

print(f'Cell {CELL}: {recipe["desc"]}')
print(f'  model={MODEL_KIND}  d={D}  L={L}  V_theta={V_THETA_KIND}(K={V_THETA_K})  M={N_REGISTERS}')
print(f'  n_attn={N_ATTN}  lambda_V={LAMBDA_V}  steps={STEPS}  batch={BATCH}')
print(f'  init_gamma={INIT_GAMMA}  fixed_gamma={FIXED_GAMMA}')


## 3. Load TinyStories

In [ ]:
from data_module import load_tiny_stories, get_batch

train_ids, val_ids = load_tiny_stories(max_train_tokens=5_000_000)
print(f'train: {len(train_ids):,} tokens   val: {len(val_ids):,} tokens')

## 4. Build model

In [ ]:
from parf.model_fock_parf import FockPARFLM, FockPARFConfig
from parf.model_structured_vtheta import (
    MixtureQuadraticVTheta, QuadraticWellVTheta,
    LowRankQuadraticVTheta, HybridQuadraticVTheta,
)
from parf.model_hybrid_fock_parf import HybridFockPARF, HybridFockPARFConfig
from parf.model_parf_sparse import SparsePARFLM, SparsePARFConfig
from sarf_mass_variant.model_sarf_mass import causal_cumulative_mean
from hybrid.model_hybrid import HybridSPLM, HSPLMConfig
import torch.nn.functional as F_torch

SCALEUP_LOGFREQ = SARF_DIR / 'scaleup' / 'results' / 'logfreq_surprisal_tinystories.npy'
DRIVE_LOGFREQ = RESULTS_ROOT / 'logfreq_surprisal_tinystories.npy'

if SCALEUP_LOGFREQ.exists():
    LOGFREQ_PATH = SCALEUP_LOGFREQ
    print(f'Using bundled logfreq: {LOGFREQ_PATH}')
elif DRIVE_LOGFREQ.exists():
    LOGFREQ_PATH = DRIVE_LOGFREQ
    print(f'Using Drive-cached logfreq: {LOGFREQ_PATH}')
else:
    counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE).astype(np.float64)
    p = (counts + 1.0) / (counts.sum() + VOCAB_SIZE)
    surprisal = (-np.log(p)).astype(np.float32)
    LOGFREQ_PATH = DRIVE_LOGFREQ
    LOGFREQ_PATH.parent.mkdir(parents=True, exist_ok=True)
    np.save(LOGFREQ_PATH, surprisal)
    print(f'Built logfreq from train_ids; saved to {LOGFREQ_PATH}')

torch.manual_seed(SEED)

gamma_kwargs = dict(init_gamma=INIT_GAMMA)
if FIXED_GAMMA is not None:
    gamma_kwargs['fixed_gamma'] = FIXED_GAMMA

parf_base_kw = dict(
    vocab_size=VOCAB_SIZE, d=D, max_len=MAX_LEN,
    L=L, v_hidden=V_HIDDEN, v_depth=V_DEPTH, dt=DT,
    v_phi_kind=V_PHI_KIND,
    v_phi_d_type=V_PHI_D_TYPE, v_phi_d_angle=V_PHI_D_ANGLE,
    v_phi_phi_hidden=V_PHI_PHI_HIDDEN,
    v_phi_theta_hidden=V_PHI_THETA_HIDDEN,
    v_phi_mlp_hidden=V_PHI_MLP_HIDDEN,
    mass_mode='logfreq',
    logfreq_path=str(LOGFREQ_PATH),
    top_k=TOP_K,
    score_head_hidden=SCORE_HEAD_HIDDEN,
    gumbel_tau_init=GUMBEL_TAU_INIT,
    gumbel_tau_min=GUMBEL_TAU_MIN,
    **gamma_kwargs,
)

if MODEL_KIND == 'hybrid_fock_parf':
    cfg = HybridFockPARFConfig(
        **parf_base_kw,
        n_registers=N_REGISTERS,
        creation_gate_hidden=CREATION_GATE_HIDDEN,
        stack_discipline=STACK_DISCIPLINE,
        n_attn=N_ATTN, n_head=N_HEAD, mlp_mult=MLP_MULT,
    )
    model = HybridFockPARF(cfg).to(device)
    print(f'Hybrid FockPARF: n_attn={N_ATTN}, L={L} FockPARF layers')
elif MODEL_KIND == 'hybrid_splm':
    cfg = HSPLMConfig(
        vocab_size=VOCAB_SIZE, d=D, max_len=MAX_LEN,
        n_attn=N_ATTN, n_splm=L, n_head=N_HEAD, mlp_mult=MLP_MULT,
        v_hidden=V_HIDDEN, v_depth=V_DEPTH, dt=DT,
        mass_mode='logfreq',
        logfreq_path=str(LOGFREQ_PATH),
        **gamma_kwargs,
    )
    model = HybridSPLM(cfg).to(device)
    print(f'Hybrid SPLM: n_attn={N_ATTN}, n_splm={L}')
elif MODEL_KIND == 'parf':
    cfg = SparsePARFConfig(**parf_base_kw)
    model = SparsePARFLM(cfg).to(device)
    print(f'SparsePARFLM: L={L}, v_hidden={V_HIDDEN}')
else:
    cfg = FockPARFConfig(
        **parf_base_kw,
        n_registers=N_REGISTERS,
        creation_gate_hidden=CREATION_GATE_HIDDEN,
        stack_discipline=STACK_DISCIPLINE,
    )
    model = FockPARFLM(cfg).to(device)
    print(f'FockPARF: L={L}, M={N_REGISTERS}')

n_total = sum(p.numel() for p in model.parameters())
n_v_theta = sum(p.numel() for p in model.V_theta.parameters())
n_v_phi = sum(p.numel() for p in model.V_phi.parameters()) if hasattr(model, 'V_phi') else 0
n_fock = model.get_register_overhead() if hasattr(model, 'get_register_overhead') else 0
print(f'params: total={n_total:,}  V_theta={n_v_theta:,}  V_phi={n_v_phi:,}  '
      f'Fock_overhead={n_fock:,}')

# ── Structured V_theta hot-swap ─────────────────────────────
if V_THETA_KIND == 'sq3':
    model.V_theta = MixtureQuadraticVTheta(d=D, K=V_THETA_K).to(device)
    n_v_theta = sum(p.numel() for p in model.V_theta.parameters())
    print(f'[SQ3] Replaced V_theta with MixtureQuadraticVTheta(K={V_THETA_K})')
    print(f'      V_theta params: {n_v_theta:,}  '
          f'(attractor centres readable analytically from model.V_theta.attractor_centres(xi))')
elif V_THETA_KIND == 'sq1':
    model.V_theta = QuadraticWellVTheta(d=D).to(device)
    n_v_theta = sum(p.numel() for p in model.V_theta.parameters())
    print(f'[SQ1] Replaced V_theta with QuadraticWellVTheta')
    print(f'      V_theta params: {n_v_theta:,}')
elif V_THETA_KIND == 'sq2':
    model.V_theta = LowRankQuadraticVTheta(d=D, rank=V_THETA_K).to(device)
    n_v_theta = sum(p.numel() for p in model.V_theta.parameters())
    print(f'[SQ2] Replaced V_theta with LowRankQuadraticVTheta(rank={V_THETA_K})')
    print(f'      V_theta params: {n_v_theta:,}')
else:  # mlp — keep original
    print(f'[MLP] Keeping original MLP V_theta (v_hidden={V_HIDDEN}, v_depth={V_DEPTH})')

HAS_STRUCTURED_VTHETA = V_THETA_KIND in ('sq1', 'sq2', 'sq3')
IS_MIXTURE = V_THETA_KIND == 'sq3'


## 5. Training loop with V_θ regularisation

In [ ]:
import math, time, json


def lr_at(step):
    if step < WARMUP:
        return LR * (step + 1) / WARMUP
    progress = (step - WARMUP) / max(STEPS - WARMUP, 1)
    return LR * 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))


def tau_at(step):
    anneal_fraction = 0.8
    warm = int((1.0 - anneal_fraction) * STEPS)
    if step < warm:
        return GUMBEL_TAU_INIT
    if step >= STEPS:
        return GUMBEL_TAU_MIN
    progress = (step - warm) / max(STEPS - warm, 1)
    return GUMBEL_TAU_INIT + (GUMBEL_TAU_MIN - GUMBEL_TAU_INIT) * min(progress, 1.0)


def forward_with_vreg(model, x, targets, lambda_v):
    """Decomposed forward for all model kinds."""
    if MODEL_KIND == 'hybrid_splm':
        logits, loss_ntp = model(x, targets)[:2]
        v_reg_value = torch.tensor(0.0, device=x.device)
        if lambda_v > 0:
            with torch.enable_grad():
                _, _, traj = model(x, targets, return_trajectory=True)
            h_L = traj[-1].to(device)
            xi = causal_cumulative_mean(h_L.detach())
            V_vals = model.V_theta(xi, h_L)
            v_reg_value = (V_vals ** 2).mean()
            loss = loss_ntp + lambda_v * v_reg_value
        else:
            loss = loss_ntp
        return logits, loss, loss_ntp, v_reg_value

    h0 = model._embed(x)
    if MODEL_KIND == 'hybrid_fock_parf':
        h_attn, _ = model._attn_stack(h0)
        h_k = model.ln_boundary(h_attn)
        h_L, _ = model._stack_forward(h_k, x, return_trajectory=False)
    else:
        h_L, _ = model._stack_forward(h0, x, return_trajectory=False)

    logits = h_L @ model.E.weight.T
    loss_ntp = F_torch.cross_entropy(
        logits.reshape(-1, model.cfg.vocab_size),
        targets.reshape(-1),
    )

    v_reg_value = torch.tensor(0.0, device=x.device)
    if lambda_v > 0:
        xi = causal_cumulative_mean(h_L.detach())
        V_vals = model.V_theta(xi, h_L)
        v_reg_value = (V_vals ** 2).mean()
        loss = loss_ntp + lambda_v * v_reg_value
    else:
        loss = loss_ntp

    return logits, loss, loss_ntp, v_reg_value


@torch.no_grad()
def evaluate():
    model.eval()
    losses = []
    for _ in range(EVAL_ITERS):
        xb, yb = get_batch(val_ids, BATCH, BLOCK, rng)
        x = torch.from_numpy(xb).to(device)
        y = torch.from_numpy(yb).to(device)
        with torch.enable_grad():
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))


HAS_GUMBEL = hasattr(model, 'set_gumbel_tau')

opt = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, betas=(0.9, 0.95), weight_decay=WD,
)
model.train()

log = []
best_ppl = float('inf')
t0 = time.time()

for step in range(STEPS):
    for g in opt.param_groups:
        g['lr'] = lr_at(step)
    if HAS_GUMBEL:
        model.set_gumbel_tau(tau_at(step))

    xb, yb = get_batch(train_ids, BATCH, BLOCK, rng)
    x = torch.from_numpy(xb).to(device)
    y = torch.from_numpy(yb).to(device)

    _, loss, loss_ntp, v_reg = forward_with_vreg(model, x, y, LAMBDA_V)

    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(
        [p for p in model.parameters() if p.requires_grad], GRAD_CLIP,
    )
    opt.step()

    if (step + 1) % LOG_INTERVAL == 0 or step == 0:
        tau_str = f'tau={tau_at(step):.3f}  ' if HAS_GUMBEL else ''
        msg = (f'[{CELL}] step {step + 1:>5}/{STEPS}  '
               f'lr={lr_at(step):.2e}  '
               f'{tau_str}'
               f'ntp={loss_ntp.item():.4f}  '
               f'v_reg={v_reg.item():.4f}  '
               f'total={loss.item():.4f}  '
               f'wall={time.time() - t0:.0f}s')
        print(msg)

    if (step + 1) % EVAL_INTERVAL == 0 or (step + 1) == STEPS:
        val_loss = evaluate()
        val_ppl = math.exp(val_loss)
        if val_ppl < best_ppl:
            best_ppl = val_ppl
        print(f'  >> val_loss={val_loss:.4f}  val_ppl={val_ppl:.2f}  '
              f'best_ppl={best_ppl:.2f}')
        log.append({
            'step': step + 1, 'val_loss': val_loss,
            'val_ppl': val_ppl, 'best_ppl': best_ppl,
            'train_loss_ntp': loss_ntp.item(),
            'v_reg': v_reg.item(), 'train_loss_total': loss.item(),
            'lambda_v': LAMBDA_V,
        })

print(f'\n[{CELL}] Training done.  total wall = {time.time() - t0:.0f}s  '
      f'final val_ppl = {log[-1]["val_ppl"]:.2f}  '
      f'best_ppl = {best_ppl:.2f}')

## 6. Save checkpoint and training log

In [ ]:
full_tag = f'tinystories_{CELL}_{MODEL_KIND}_d{D}_L{L}'
if N_REGISTERS > 0:
    full_tag += f'_M{N_REGISTERS}'
full_tag += f'_seed{SEED}'

ckpt_path = RUN_DIR / f'{full_tag}_ckpt_latest.pt'
torch.save({
    'model_state_dict': model.state_dict(),
    'config': vars(cfg) if hasattr(cfg, '__dict__') else str(cfg),
    'recipe': recipe,
    'best_ppl': best_ppl,
    'step': STEPS,
}, ckpt_path)
print(f'checkpoint saved: {ckpt_path}')

log_path = RUN_DIR / f'{full_tag}_training_log.jsonl'
with open(log_path, 'w') as f:
    for entry in log:
        f.write(json.dumps(entry) + '\n')
print(f'training log saved: {log_path}')

## 7. Training curve

In [ ]:
import matplotlib.pyplot as plt

steps_arr = [e['step'] for e in log]
ppls = [e['val_ppl'] for e in log]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(steps_arr, ppls, 'o-', label=f'{CELL}: {recipe["desc"]}', linewidth=2)

ax.axhline(7.81, color='red', linestyle='--', alpha=0.6, label='Attn baseline (7.81)')
ax.axhline(26.42, color='orange', linestyle='--', alpha=0.6, label='PARF P10g unreg (26.42)')
ax.axhline(8.85, color='green', linestyle='--', alpha=0.6, label='SPLM em_ln MPS (8.85)')

ax.set_xlabel('Training step')
ax.set_ylabel('Val PPL')
ax.set_title(f'TinyStories: {CELL} ({recipe["desc"]})')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RUN_DIR / f'{full_tag}_training.png', dpi=120)
plt.show()
print(f'best PPL = {best_ppl:.2f}')

## 8. V_θ landscape diagnostics

In [ ]:
v_samples = []
model.eval()
with torch.no_grad():
    for _ in range(10):
        xb, _ = get_batch(val_ids, BATCH, BLOCK, rng)
        x = torch.from_numpy(xb).to(device)
        with torch.enable_grad():
            out = model(x, return_trajectory=True)
        traj = out[2]
        h_L = traj[-1].to(device)
        xi = causal_cumulative_mean(h_L)
        V_vals = model.V_theta(xi, h_L).detach().cpu().numpy().ravel()
        v_samples.append(V_vals)

V_all = np.concatenate(v_samples)
print(f'V_theta on real trajectories ({CELL}):')
print(f'  mean   = {V_all.mean():.2f}')
print(f'  std    = {V_all.std():.2f}')
print(f'  min    = {V_all.min():.2f}')
print(f'  max    = {V_all.max():.2f}')
print(f'  range  = {V_all.max() - V_all.min():.2f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(V_all, bins=100, color='#3a6ea5', alpha=0.7, edgecolor='white')
ax.axvline(0, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('V_\u03b8(\u03be, h)')
ax.set_ylabel('count')
ax.set_title(f'{CELL} V_\u03b8 distribution ({recipe["desc"]})')
plt.tight_layout()
plt.savefig(RUN_DIR / f'{full_tag}_v_theta_hist.png', dpi=120)
plt.show()

landscape_stats = {
    'mean': float(V_all.mean()), 'std': float(V_all.std()),
    'min': float(V_all.min()), 'max': float(V_all.max()),
    'range': float(V_all.max() - V_all.min()),
}
with open(RUN_DIR / f'{full_tag}_landscape_stats.json', 'w') as f:
    json.dump(landscape_stats, f, indent=2)
print(f'landscape stats saved')

# ── Analytical attractor centres (structured V_theta only) ──
if IS_MIXTURE:
    print()
    print('=== Analytical attractor centres (SQ3 MixtureQuadraticVTheta) ===')
    model.eval()
    from transformers import GPT2Tokenizer
    try:
        tok = GPT2Tokenizer.from_pretrained('gpt2')
        tok.pad_token = tok.eos_token
    except Exception:
        tok = None

    # Use a few fixed prompts to extract xi, then read basin centres
    prompts = [
        'Once upon a time',
        'The cat sat on',
        'In a kingdom far away',
        'She opened the door and',
    ]
    with torch.no_grad():
        all_centres = []
        for prompt in prompts:
            if tok is not None:
                ids = tok.encode(prompt, return_tensors='pt').to(device)
            else:
                xb, _ = get_batch(val_ids, 1, BLOCK, rng)
                ids = torch.from_numpy(xb).to(device)[:, :8]
            h0 = model._embed(ids)
            if MODEL_KIND == 'hybrid_fock_parf':
                h_attn, _ = model._attn_stack(h0)
                h_k = model.ln_boundary(h_attn)
                h_L, _ = model._stack_forward(h_k, ids, return_trajectory=False)
            elif MODEL_KIND == 'hybrid_splm':
                h_L = model(ids)[-1][-1] if hasattr(model, '_stack_forward') else h0
            else:
                h_L, _ = model._stack_forward(h0, ids, return_trajectory=False)
            xi = causal_cumulative_mean(h_L.detach())
            # centres shape: (B, T, K, d)
            centres = model.V_theta.attractor_centres(xi)
            # Take the last token position's centres for the first batch item
            c_last = centres[0, -1, :, :]  # (K, d)
            all_centres.append(c_last.cpu())
            if tok is not None:
                print(f'  Prompt: {prompt!r}')
                for k in range(V_THETA_K):
                    # Project to vocab — find nearest token
                    scores = (c_last[k] @ model.E.weight.T).cpu()
                    top5 = scores.topk(5).indices.tolist()
                    top5_toks = [tok.decode([t]) for t in top5]
                    print(f'    Basin {k}: {top5_toks}')
                print()

    # Save attractor centres summary
    centres_summary = {}
    for i, prompt in enumerate(prompts):
        centres_summary[prompt] = all_centres[i].tolist()
    with open(RUN_DIR / f'{full_tag}_attractor_centres.json', 'w') as f:
        json.dump(centres_summary, f, indent=2)
    print(f'  attractor centres saved to {full_tag}_attractor_centres.json')


## 9. Cross-cell comparison dashboard

In [ ]:
results = {}
for cell_name in ('S1', 'S2', 'S3', 'S4'):
    cell_dir = RESULTS_ROOT / cell_name / f'seed{SEED}'
    if not cell_dir.exists():
        results[cell_name] = None
        continue
    logs = sorted(cell_dir.glob('*_training_log.jsonl'))
    if not logs:
        results[cell_name] = None
        continue
    rows = [json.loads(line) for line in logs[-1].read_text().splitlines()]
    last = rows[-1] if rows else None
    best = min(r['val_ppl'] for r in rows) if rows else None
    ls_files = sorted(cell_dir.glob('*_landscape_stats.json'))
    ls = None
    if ls_files:
        ls = json.loads(ls_files[-1].read_text())
    results[cell_name] = {
        'desc': SCALE_RECIPES[cell_name]['desc'],
        'val_ppl': last['val_ppl'] if last else None,
        'best_ppl': best,
        'landscape': ls,
    }

print(f'{"Cell":<6} {"Description":<50} {"best PPL":>10} {"final PPL":>10} '
      f'{"V range":>8}')
print('-' * 95)

print(f'{"":<6} {"Attn baseline (8L GPT-2)":<50} {"7.81":>10} '
      f'{"-":>10} {"-":>8}')
print(f'{"":<6} {"SPLM em_ln (MPS)":<50} {"8.85":>10} '
      f'{"-":>10} {"-":>8}')
print(f'{"":<6} {"PARF P10g (unreg, 16k steps)":<50} {"26.42":>10} '
      f'{"-":>10} {"-":>8}')
# v2 reference results (MLP V_theta, 16k steps) — fill in once v2 runs complete
V2_REF = {
    'S1': {'best_ppl': None,   'v_range': None},    # Reg PARF MLP V_theta (not yet run)
    'S2': {'best_ppl': 27.851, 'v_range': 61.336},  # Reg FockPARF d=256 L=8 M=32 MLP V_theta
    'S3': {'best_ppl': 8.010,  'v_range': 3.338},   # Hybrid FockPARF+Attn d=256 L=4+4 MLP V_theta
    'S4': {'best_ppl': None,   'v_range': None},    # Hybrid SPLM+Attn MLP V_theta (not yet run)
}
# Try to load v2 results from GDrive automatically
_v2_root = GDRIVE_OUT.parent / 'semsimula_tinystories_v2'
for _cn in ('S1', 'S2', 'S3', 'S4'):
    _v2_dir = _v2_root / _cn / f'seed{SEED}'
    if _v2_dir.exists():
        _logs = sorted(_v2_dir.glob('*_training_log.jsonl'))
        if _logs:
            _rows = [json.loads(l) for l in _logs[-1].read_text().splitlines()]
            V2_REF[_cn]['best_ppl'] = min(r['val_ppl'] for r in _rows) if _rows else None
        _ls_files = sorted(_v2_dir.glob('*_landscape_stats.json'))
        if _ls_files:
            _ls = json.loads(_ls_files[-1].read_text())
            V2_REF[_cn]['v_range'] = _ls.get('range')

print(f'{"":6} {"Attn baseline (8L GPT-2)":<50} {"7.81":>10} {"-":>10} {"-":>8}')
print(f'{"":6} {"PARF P10g (unreg, 16k steps)":<50} {"26.42":>10} {"-":>10} {"-":>8}')
print(f'{"":6} {"TinyShakespeare SQ3 PARF (4k, d=128)":<50} {"184.5":>10} {"-":>10} {"-":>8}')
for _cn, _label in [
    ('S1', 'v2 Reg PARF MLP (16k)'),
    ('S2', 'v2 Reg FockPARF MLP (16k)'),
    ('S3', 'v2 Hybrid FockPARF+Attn MLP (16k)'),
    ('S4', 'v2 Hybrid SPLM+Attn MLP (16k)'),
]:
    _r = V2_REF[_cn]
    _p = f"{_r['best_ppl']:.1f}" if _r['best_ppl'] else '—'
    _v = f"{_r['v_range']:.1f}" if _r['v_range'] else '—'
    print(f'{"":6} {_label:<50} {_p:>10} {"—":>10} {_v:>8}  (v2 MLP ref)')
      f'{"-":>10} {"69.1":>8}    # under-reg')
      f'{"-":>10} {"7.2":>8}    # V_theta zeroed')
print('-' * 95)

for cell_name, r in results.items():
    if r is None:
        desc = SCALE_RECIPES[cell_name]['desc']
        print(f'{cell_name:<6} {desc:<50} {"\u2014":>10} {"\u2014":>10} '
              f'{"\u2014":>8}    (not run)')
        continue
    desc = r['desc']
    ppl_final = f"{r['val_ppl']:.1f}" if r['val_ppl'] else '\u2014'
    ppl_best = f"{r['best_ppl']:.1f}" if r['best_ppl'] else '\u2014'
    v_range = f"{r['landscape']['range']:.1f}" if r['landscape'] else '\u2014'
    print(f'{cell_name:<6} {desc:<50} {ppl_best:>10} {ppl_final:>10} '
          f'{v_range:>8}')

print(f'\n\u2192  Key question: does FockPARF (S2) beat PARF (S1) on TinyStories?')
print(f'\u2192  Does Hybrid FockPARF+Attn (S3) beat Hybrid SPLM+Attn (S4)?')